# Tunable Research Parameters — The Complete Guide

This notebook documents **every tunable hyperparameter** in the Assistant Axis research methodology. Each parameter is categorized by pipeline stage, with its default value, valid range, impact level, and code showing exactly how to change it.

**Use this notebook to:**
- Understand what knobs you can turn and what they do
- Plan ablation studies by systematically varying parameters
- Reproduce results with full parameter transparency
- Explore how sensitive the axis is to methodological choices

---

## Table of Contents

1. [Parameter Overview — Summary Table](#1-overview)
2. [Response Generation Parameters](#2-generation)
3. [Activation Extraction Parameters](#3-extraction)
4. [Judge Scoring Parameters](#4-scoring)
5. [Vector Computation Parameters](#5-vectors)
6. [Axis Computation Parameters](#6-axis)
7. [PCA Analysis Parameters](#7-pca)
8. [Steering & Capping Parameters](#8-steering)
9. [Projection Parameters](#9-projection)
10. [Model Configuration Parameters](#10-model-config)
11. [Ablation Study Templates](#11-ablation)

---

## 1. Parameter Overview — Summary Table <a id="1-overview"></a>

All 51 tunable parameters at a glance, sorted by impact level.

| # | Parameter | Default | Stage | Impact | How to Change |
|---|-----------|---------|-------|--------|---------------|
| 1 | `intervention_type` | `"addition"` | Steering | CRITICAL | Constructor arg |
| 2 | `coefficients` | `1.0` | Steering | CRITICAL | Constructor arg |
| 3 | `cap_thresholds` (tau) | Calibrated | Steering | CRITICAL | Constructor arg |
| 4 | `layer_indices` | `-1` | Steering | CRITICAL | Constructor arg |
| 5 | `score_threshold` | `3` | Vectors | CRITICAL | Edit `4_vectors.py` |
| 6 | `temperature` | `0.7` | Generation | HIGH | CLI `--temperature` |
| 7 | `question_count` | `240` | Generation | HIGH | CLI `--question_count` |
| 8 | `max_tokens` | `512` | Generation | HIGH | CLI `--max_tokens` |
| 9 | `layers` | `"all"` | Extraction | HIGH | CLI `--layers` |
| 10 | `max_length` | `2048` | Extraction | HIGH | CLI `--max_length` |
| 11 | `judge_model` | `"gpt-4.1-mini"` | Scoring | HIGH | CLI `--judge_model` |
| 12 | `min_count` | `50` | Vectors | HIGH | CLI `--min_count` |
| 13 | `aggregation` | `mean` | Axis | HIGH | Edit `5_axis.py` |
| 14 | `scaler` | `None` | PCA | HIGH | Function arg |
| 15 | `target_layer` | Model-specific | Config | HIGH | Edit `models.py` |
| 16 | `positions` | `"all"` | Steering | HIGH | Constructor arg |
| 17 | `model` | Required | Generation | HIGH | CLI `--model` |
| 18 | `top_p` | `0.9` | Generation | MEDIUM | CLI `--top_p` |
| 19 | `prompt_indices` | `[0,1,2,3,4]` | Generation | MEDIUM | Constructor arg |
| 20 | `batch_size` (extraction) | `16` | Extraction | MEDIUM | CLI `--batch_size` |
| 21 | `thinking` | `False` | Extraction | MEDIUM | CLI `--thinking` |
| 22 | `normalize` | `True` | Projection | MEDIUM | Function arg |
| 23 | `batch_size` (judge) | `50` | Scoring | LOW | CLI `--batch_size` |
| 24 | `requests_per_second` | `100` | Scoring | LOW | CLI `--requests_per_second` |
| 25 | `debug` | `False` | Steering | LOW | Constructor arg |
| 26 | `epsilon` | `1e-8` | Projection | LOW | Edit source |

Plus ~25 additional implicit/hardcoded parameters documented in detail below.

---

## 2. Response Generation Parameters <a id="2-generation"></a>

These parameters control **how responses are generated** in Pipeline Step 1. They determine the diversity, length, and quality of the raw data that everything else builds on.

**Source files:** `pipeline/1_generate.py`, `assistant_axis/generation.py`

In [ ]:
# ============================================================
# 2.1  MODEL SELECTION
# ============================================================
# Impact: HIGH — Different models have different activation spaces, layer counts, and behaviors.
# The axis computed from one model is NOT transferable to another.

# CLI usage:
# python pipeline/1_generate.py --model "google/gemma-2-27b-it"
# python pipeline/1_generate.py --model "Qwen/Qwen3-32B"
# python pipeline/1_generate.py --model "meta-llama/Llama-3.3-70B-Instruct"

# In code:
from assistant_axis.generation import VLLMGenerator

generator = VLLMGenerator(
    model_name="google/gemma-2-27b-it",  # <-- CHANGE THIS to switch models
)

# Supported models with pre-configured settings:
from assistant_axis.models import MODEL_CONFIGS
for name, cfg in MODEL_CONFIGS.items():
    print(f"  {name}: target_layer={cfg['target_layer']}, total_layers={cfg['total_layers']}")

In [ ]:
# ============================================================
# 2.2  SAMPLING PARAMETERS (temperature, top_p, max_tokens)
# ============================================================
# These control the randomness, diversity, and length of generated responses.

# Impact:
#   temperature — HIGH: controls response diversity. Higher = more varied role-playing.
#   top_p       — MEDIUM: nucleus sampling cutoff. Works with temperature.
#   max_tokens  — HIGH: response length cap. Too short = truncated responses.

# --- CLI usage ---
# python pipeline/1_generate.py --model "Qwen/Qwen3-32B" \
#     --temperature 0.7 \
#     --top_p 0.9 \
#     --max_tokens 512

# --- In code ---
generator = VLLMGenerator(
    model_name="Qwen/Qwen3-32B",
    temperature=0.7,    # DEFAULT: 0.7  | Range: 0.0 (deterministic) to 2.0 (very random)
    top_p=0.9,          # DEFAULT: 0.9  | Range: 0.0 to 1.0 (1.0 = no filtering)
    max_tokens=512,     # DEFAULT: 512  | Range: 1 to model's max context length
)

# --- What happens at the extremes ---
# temperature=0.0: Greedy decoding, every response is identical for same prompt
# temperature=2.0: Very noisy, responses may be incoherent
# top_p=0.1: Very focused, only top 10% probability tokens sampled
# top_p=1.0: Full distribution, no nucleus filtering
# max_tokens=64: Very short responses, may lose role-playing nuance
# max_tokens=2048: Long responses, more data per response but slower

# --- Ablation idea ---
# Compare axis quality across temperature values:
for temp in [0.3, 0.5, 0.7, 1.0, 1.5]:
    print(f"  temperature={temp}: {'deterministic' if temp < 0.3 else 'diverse' if temp < 1.0 else 'noisy'}")

In [ ]:
# ============================================================
# 2.3  DATA VOLUME PARAMETERS (question_count, prompt_indices)
# ============================================================
# These control HOW MUCH data is generated per role.
# Total responses per role = len(prompt_indices) x question_count

# Impact:
#   question_count  — HIGH: sample size per role. Directly affects vector quality.
#   prompt_indices  — MEDIUM: which system prompt variants to use (5 available per role).

# --- CLI usage ---
# python pipeline/1_generate.py --model "Qwen/Qwen3-32B" --question_count 240

# --- In code (question_count) ---
from assistant_axis.generation import RoleResponseGenerator

role_gen = RoleResponseGenerator(
    model_name="Qwen/Qwen3-32B",
    roles_dir="data/roles/instructions",
    output_dir="outputs/responses",
    questions_file="data/extraction_questions.jsonl",
    question_count=240,  # DEFAULT: 240 | Range: 1 to 240 (total available questions)
)

# --- In code (prompt_indices) ---
# By default, all 5 variants are used: [0, 1, 2, 3, 4]
# Each role JSON has 5 system prompt variants in role["instruction"]

role_gen = RoleResponseGenerator(
    model_name="Qwen/Qwen3-32B",
    roles_dir="data/roles/instructions",
    output_dir="outputs/responses",
    questions_file="data/extraction_questions.jsonl",
    question_count=240,
    prompt_indices=[0, 1, 2],  # Only use first 3 variants (instead of all 5)
)

# --- Total responses per role ---
# Default: 5 variants x 240 questions = 1,200 responses per role
# With prompt_indices=[0]: 1 variant x 240 questions = 240 responses per role
# With question_count=50, prompt_indices=[0,1]: 2 x 50 = 100 responses per role

for n_prompts in [1, 3, 5]:
    for n_questions in [50, 120, 240]:
        total = n_prompts * n_questions
        print(f"  {n_prompts} prompts x {n_questions} questions = {total} responses/role")

In [ ]:
# ============================================================
# 2.4  INFRASTRUCTURE PARAMETERS (tensor_parallel, gpu_memory, max_model_len)
# ============================================================
# These don't affect research results, but control GPU utilization and throughput.

# Impact: LOW on results, HIGH on runtime

# --- CLI usage ---
# python pipeline/1_generate.py --model "Qwen/Qwen3-32B" \
#     --tensor_parallel_size 2 \
#     --gpu_memory_utilization 0.95 \
#     --max_model_len 2048

# --- In code ---
generator = VLLMGenerator(
    model_name="Qwen/Qwen3-32B",
    tensor_parallel_size=2,         # DEFAULT: None (auto-detect) | Range: 1 to num_gpus
    gpu_memory_utilization=0.95,    # DEFAULT: 0.95 (pipeline), 0.9 (library) | Range: 0.0-1.0
    max_model_len=2048,             # DEFAULT: 2048 | Range: 1 to model's actual max
)

# tensor_parallel_size: Shard model across N GPUs. Set to match your hardware.
# gpu_memory_utilization: How much VRAM to use. 0.95 = aggressive, 0.8 = conservative.
# max_model_len: Max context window for vLLM. Affects KV cache size.

# Note: The pipeline also supports multi-WORKER parallelism (multiple vLLM instances)
# when total_gpus > tensor_parallel_size. This distributes ROLES across workers.
print("Infrastructure parameters affect speed, not research quality.")

---

## 3. Activation Extraction Parameters <a id="3-extraction"></a>

These parameters control **how hidden states are captured** from the model in Pipeline Step 2. They determine which layers, which tokens, and how activations are aggregated.

**Source files:** `pipeline/2_activations.py`, `assistant_axis/internals/activations.py`, `assistant_axis/internals/spans.py`

In [ ]:
# ============================================================
# 3.1  LAYER SELECTION
# ============================================================
# Impact: HIGH — Different layers encode different information.
# Early layers: syntax/tokens. Middle layers: semantics/persona. Late layers: output formatting.

# --- CLI usage ---
# python pipeline/2_activations.py --model "Qwen/Qwen3-32B" --layers all
# python pipeline/2_activations.py --model "Qwen/Qwen3-32B" --layers 32        # single layer
# python pipeline/2_activations.py --model "Qwen/Qwen3-32B" --layers 0,16,32,48,63  # specific layers

# --- In code (ActivationExtractor) ---
from assistant_axis.internals import ActivationExtractor

# Extract all layers — produces (n_layers, n_tokens, hidden_dim)
# activations = extractor.full_conversation(conversation, layer=None)

# Extract single layer — produces (n_tokens, hidden_dim)
# activations = extractor.full_conversation(conversation, layer=32)

# Extract specific layers — produces dict {layer_idx: (n_tokens, hidden_dim)}
# activations = extractor.full_conversation(conversation, layer=[10, 20, 32, 50])

# --- Recommended target layers (from models.py) ---
target_layers = {
    "google/gemma-2-27b-it": 22,              # 22 of 46 (48% depth)
    "Qwen/Qwen3-32B": 32,                     # 32 of 64 (50% depth)
    "meta-llama/Llama-3.3-70B-Instruct": 40,  # 40 of 80 (50% depth)
}
# Rule of thumb: middle layers (~50% depth) capture persona best.
# The paper found cosine similarity with PC1 > 0.71 at middle layers.

# --- Ablation idea ---
# Compute axis at every layer and measure effectiveness:
print("Layer sweep: compute axis at layers [10, 20, 30, 40, 50, 60] and compare steering quality")

In [ ]:
# ============================================================
# 3.2  BATCH SIZE AND SEQUENCE LENGTH
# ============================================================
# Impact:
#   batch_size  — MEDIUM: affects GPU memory usage, not research results (numerically identical)
#   max_length  — HIGH: sequences longer than this are TRUNCATED, losing activation data

# --- CLI usage ---
# python pipeline/2_activations.py --model "Qwen/Qwen3-32B" \
#     --batch_size 16 \
#     --max_length 2048

# --- In code (batch_conversations) ---
# From assistant_axis/internals/activations.py, batch_conversations():
#
# batch_activations, batch_metadata = extractor.batch_conversations(
#     conversations,
#     layer=None,           # Which layers
#     max_length=2048,      # DEFAULT: 2048 | Sequences exceeding this are truncated
# )

# --- What truncation does ---
# If a conversation tokenizes to 3000 tokens but max_length=2048:
#   - Only the first 2048 tokens are processed
#   - Later assistant turns may be partially or fully lost
#   - batch_metadata["truncated_lengths"] records original vs truncated lengths

# --- Recommendations ---
# max_length=2048: Safe default for single-turn responses (512 tokens max)
# max_length=4096: Needed for multi-turn conversations
# max_length=1024: Faster but risks truncating longer responses

print("batch_size=16 (default) is a good balance for most GPUs")
print("max_length=2048 handles all single-turn responses (max 512 generated tokens)")

In [ ]:
# ============================================================
# 3.3  TOKEN AGGREGATION METHOD (implicit — hardcoded)
# ============================================================
# Impact: HIGH — This is one of the most fundamental methodological choices.
# How you aggregate token-level activations into a single vector per turn matters.

# CURRENT BEHAVIOR (hardcoded in spans.py and 2_activations.py):
#   1. For each assistant turn, extract activations at ALL response tokens
#   2. Compute the MEAN across all tokens in the span
#   3. If multi-turn, compute the MEAN across all assistant turns
#   Result: one (n_layers, hidden_dim) vector per response

# --- Where this is defined ---
# File: assistant_axis/internals/spans.py, map_spans():
#
#   turn_acts = batch_activations[:, conv_idx, start:end, :]  # (n_layers, n_tokens, hidden)
#   turn_mean = turn_acts.mean(dim=1)                          # (n_layers, hidden)
#
# File: pipeline/2_activations.py, lines 113-124:
#
#   # For single-turn: take turn index 1 (the assistant response)
#   # For multi-turn:  take all odd-indexed turns (assistant turns)
#   turn_activations = [per_turn[i] for i in range(1, len(per_turn), 2)]
#   response_activation = torch.stack(turn_activations).mean(dim=0)

# --- Alternative aggregation methods you could try ---
# 1. LAST TOKEN ONLY: Use activation at the final token of the response
#    Change: turn_acts[:, -1, :] instead of turn_acts.mean(dim=1)
#    Rationale: Last token may carry most semantic summary
#
# 2. MAX POOLING: Take element-wise max across tokens
#    Change: turn_acts.max(dim=1).values
#    Rationale: Captures strongest activation per dimension
#
# 3. WEIGHTED MEAN: Weight later tokens more heavily
#    Change: Apply positional weights before averaging
#    Rationale: Later tokens may better reflect the role after "warming up"
#
# 4. EXCLUDE CODE BLOCKS: Already implemented as map_spans_no_code()
#    Use SpanMapper.map_spans_no_code() instead of map_spans()
#    Rationale: Code syntax doesn't carry persona signal

print("Current: mean over all response tokens, mean over all assistant turns")
print("To change: modify spans.py map_spans() or pipeline/2_activations.py")

In [ ]:
# ============================================================
# 3.4  QWEN THINKING MODE
# ============================================================
# Impact: MEDIUM — Only affects Qwen models. Controls whether thinking tokens
# (from Qwen's chain-of-thought) are included in activation extraction.

# --- CLI usage ---
# python pipeline/2_activations.py --model "Qwen/Qwen3-32B" --thinking true

# --- In code ---
# Passed as enable_thinking in chat_kwargs through ConversationEncoder:
#
# encoder = ConversationEncoder(tokenizer, model_name="Qwen/Qwen3-32B")
# full_ids, spans = encoder.build_turn_spans(conversation, enable_thinking=False)
#
# When thinking=False (default):
#   - Qwen's <think>...</think> tokens are EXCLUDED from spans
#   - Only the final response tokens are used for activation extraction
#
# When thinking=True:
#   - Thinking tokens ARE included in the spans
#   - Activations during "reasoning" are mixed with "response" activations

print("thinking=False (default): exclude chain-of-thought tokens")
print("thinking=True: include them — may dilute persona signal with reasoning")

---

## 4. Judge Scoring Parameters <a id="4-scoring"></a>

These parameters control **how responses are evaluated** for role adherence in Pipeline Step 3. The judge determines which responses are "good enough" to include in vector computation.

**Source files:** `pipeline/3_judge.py`, `assistant_axis/judge.py`

In [ ]:
# ============================================================
# 4.1  JUDGE MODEL SELECTION
# ============================================================
# Impact: HIGH — Different judge models may score differently, changing which
# responses are included in vector computation. This is a key methodological choice.

# --- CLI usage ---
# python pipeline/3_judge.py --responses_dir outputs/responses --judge_model gpt-4.1-mini

# --- In code ---
from assistant_axis.judge import score_responses_sync

scores = score_responses_sync(
    responses=[{"question": "...", "response": "..."}],
    eval_prompt_template="...",
    judge_model="gpt-4.1-mini",  # DEFAULT: "gpt-4.1-mini"
    # Alternatives: "gpt-4", "gpt-4-turbo", "gpt-4o", "gpt-4.1"
)

# --- Trade-offs ---
# gpt-4.1-mini: Cheap, fast, good enough for most scoring (used in the paper)
# gpt-4:        Most capable, but expensive and slower
# gpt-4o:       Fast and capable, good balance
#
# The paper validated judge agreement with human annotators:
# 91.6% agreement on harmfulness evaluation (200 samples)

print("Judge model choice affects which responses pass the score=3 filter")
print("Consider running a small validation set with multiple judges to check consistency")

In [ ]:
# ============================================================
# 4.2  JUDGE API PARAMETERS (rate limiting, batch size, temperature)
# ============================================================
# Impact: LOW on results (same scores, just different throughput)
# Exception: judge temperature is hardcoded and COULD affect scoring consistency.

# --- CLI usage ---
# python pipeline/3_judge.py --responses_dir outputs/responses \
#     --batch_size 50 \
#     --requests_per_second 100 \
#     --max_tokens 10

# --- In code ---
scores = score_responses_sync(
    responses=[{"question": "...", "response": "..."}],
    eval_prompt_template="...",
    judge_model="gpt-4.1-mini",
    max_tokens=10,              # DEFAULT: 10  | Judge only needs to output a single digit
    batch_size=50,              # DEFAULT: 50  | Concurrent API requests per batch
    requests_per_second=100,    # DEFAULT: 100 | Rate limiter tokens/second
)

# --- Judge Temperature (HARDCODED) ---
# File: assistant_axis/judge.py, call_judge_single():
#   response = await client.chat.completions.create(
#       model=model,
#       messages=[{"role": "user", "content": prompt}],
#       max_completion_tokens=max_tokens,
#       temperature=1,  # <-- HARDCODED at 1
#   )
#
# To change: edit judge.py line ~111
# temperature=0: Deterministic scoring (always same score for same input)
# temperature=1: Standard (slight variation, current default)

# --- Score Parsing ---
# parse_judge_score() extracts first digit 0-3 from judge response using regex:
#   match = re.search(r'\b(\d+)\b', response_text)
# Returns None if no valid score found (response is discarded)

print("API parameters affect throughput, not scoring quality")
print("Judge temperature=1 is hardcoded — consider temperature=0 for reproducibility")

---

## 5. Vector Computation Parameters <a id="5-vectors"></a>

These parameters control **how per-role activation vectors are computed** in Pipeline Step 4. This is where quality filtering happens — only the best role-playing responses contribute to each role's vector.

**Source file:** `pipeline/4_vectors.py`

In [ ]:
# ============================================================
# 5.1  SCORE THRESHOLD (which responses to include)
# ============================================================
# Impact: CRITICAL — This determines what counts as "fully role-playing."
# Only responses meeting this threshold contribute to the role vector.

# CURRENT BEHAVIOR (hardcoded in pipeline/4_vectors.py):
#   if key in scores and scores[key] == 3:   # <-- ONLY score=3 responses
#       filtered_activations.append(activations[key])

# --- To change the threshold ---
# Edit pipeline/4_vectors.py, change the comparison:
#   scores[key] == 3    →  scores[key] >= 2   (include partial role-playing)
#   scores[key] == 3    →  scores[key] >= 1   (include any role attempt)

# --- Impact analysis ---
# score=3 only (default): Strictest. Uses only responses where model FULLY plays the role.
#   Pro: Cleanest signal, role vectors represent pure role-playing
#   Con: Fewer samples per role (some roles may not reach min_count)
#
# score>=2: Includes partial role-playing (model identifies as AI but shows traits)
#   Pro: More samples, more roles pass min_count
#   Con: Diluted signal — vectors mix role-playing with assistant behavior
#
# score>=1: Very permissive
#   Pro: Maximum sample count
#   Con: Very noisy vectors, axis quality likely degrades

# --- How many responses typically score 3? ---
# This varies by role and model:
# Easy roles (pirate, detective): ~80-90% score 3
# Hard roles (abstract concepts): ~30-50% score 3
# Default assistant: uses ALL activations regardless of score

print("score_threshold=3 (default): only fully role-playing responses")
print("To change: edit pipeline/4_vectors.py line ~53")

In [ ]:
# ============================================================
# 5.2  MINIMUM COUNT THRESHOLD
# ============================================================
# Impact: HIGH — Roles with fewer than min_count score=3 responses are EXCLUDED
# from the axis computation entirely. This is a quality gate.

# --- CLI usage ---
# python pipeline/4_vectors.py --activations_dir outputs/activations \
#     --scores_dir outputs/scores --min_count 50

# --- In code (from pipeline/4_vectors.py) ---
# min_count = args.min_count  # DEFAULT: 50
#
# if len(filtered_activations) < min_count:
#     print(f"Skipping {role}: only {len(filtered)} score=3 samples (need {min_count})")
#     continue

# --- Trade-offs ---
# min_count=50 (default): Moderate. Requires 50+ fully role-playing responses.
#   With 1200 responses per role, this means ~4% must score 3.
#
# min_count=100: Stricter. More reliable vectors but fewer roles survive.
# min_count=20:  Lenient. More roles but noisier vectors.
# min_count=1:   Extreme. Even a single score=3 response produces a vector.

# --- Default role handling ---
# Default roles (identified by "default" in the name) bypass this check entirely.
# They use ALL activations regardless of score, because the "default" behavior
# IS the assistant — there's no role to judge adherence to.

# --- How many roles survive at different thresholds? ---
# This depends on model and role difficulty. Typical numbers:
thresholds = [10, 20, 50, 100, 200, 500]
for t in thresholds:
    print(f"  min_count={t:>4d}: stricter filtering, fewer but higher-quality role vectors")

---

## 6. Axis Computation Parameters <a id="6-axis"></a>

These parameters control **how the final axis is computed** from per-role vectors in Pipeline Step 5. This is the simplest step but has fundamental methodological choices.

**Source files:** `pipeline/5_axis.py`, `assistant_axis/axis.py`

In [ ]:
# ============================================================
# 6.1  AGGREGATION METHOD
# ============================================================
# Impact: HIGH — How individual role vectors are combined into the axis.

# CURRENT BEHAVIOR (from axis.py compute_axis and pipeline/5_axis.py):
#
#   def compute_axis(role_activations, default_activations):
#       role_mean = role_activations.mean(dim=0)
#       default_mean = default_activations.mean(dim=0)
#       return default_mean - role_mean
#
# Both default and role vectors are aggregated by MEAN.

# --- Alternative aggregation methods ---
import torch

# 1. MEDIAN (more robust to outliers):
#    role_median = torch.median(torch.stack(role_vectors), dim=0).values
#    default_median = torch.median(torch.stack(default_vectors), dim=0).values
#    axis = default_median - role_median

# 2. TRIMMED MEAN (exclude top/bottom 10%):
#    sorted_vecs = torch.sort(torch.stack(role_vectors), dim=0).values
#    n = len(role_vectors)
#    trim = int(n * 0.1)
#    role_trimmed = sorted_vecs[trim:n-trim].mean(dim=0)

# 3. WEIGHTED MEAN (weight by role "extremeness"):
#    First compute PCA, then weight roles by distance from centroid
#    This would emphasize the most distinctive roles

# 4. ROLE SUBSETS (exclude certain categories):
from assistant_axis.axis import aggregate_role_vectors
# role_vectors_dict = {"pirate": vec1, "doctor": vec2, "default_1": vec3, ...}
# Exclude specific roles:
# role_mean = aggregate_role_vectors(role_vectors_dict, exclude=["pirate", "ninja"])

print("Current: simple mean of all role vectors and all default vectors")
print("The axis = default_mean - role_mean")
print("To change aggregation: edit axis.py compute_axis() or pipeline/5_axis.py")

In [ ]:
# ============================================================
# 6.2  DEFAULT VS ROLE CLASSIFICATION
# ============================================================
# Impact: MEDIUM — How the pipeline decides which vectors are "default" vs "role."

# CURRENT BEHAVIOR (from pipeline/5_axis.py):
#   if "default" in role_name or data["type"] == "mean":
#       default_vectors.append(data["vector"])
#   else:
#       role_vectors.append(data["vector"])

# The classification is based on:
# 1. Role name contains "default" (e.g., "default", "default_neutral", "default_helpful")
# 2. Vector type is "mean" (set in 4_vectors.py for default roles)
#    vs. "pos_3" (set for regular roles with score=3 filtering)

# --- What if you want to treat some roles as "default-adjacent"? ---
# Example: Include "reviewer", "consultant", "evaluator" as default behavior
# These are very assistant-like roles that might better represent the default persona.
#
# To do this, edit 5_axis.py to expand the default classification:
#   assistant_adjacent = {"reviewer", "consultant", "evaluator", "analyst"}
#   if "default" in role_name or role_name in assistant_adjacent:
#       default_vectors.append(...)

# --- Or exclude certain roles from the role mean ---
# Example: Remove roles that are too similar to the default (low contrast)
#   exclude_roles = {"reviewer", "consultant"}  # too assistant-like
#   if role_name not in exclude_roles:
#       role_vectors.append(...)

print("Default identification: 'default' in name OR type=='mean'")
print("All other roles contribute to the role mean")

---

## 7. PCA Analysis Parameters <a id="7-pca"></a>

These parameters control **how PCA is performed** on the role vectors. PCA validates whether the axis captures the dominant direction of variation.

**Source file:** `assistant_axis/pca.py`

In [ ]:
# ============================================================
# 7.1  SCALER CHOICE (preprocessing before PCA)
# ============================================================
# Impact: HIGH — The scaler determines how data is preprocessed before PCA.
# Different scalers yield different PCA results and different interpretations.

from assistant_axis.pca import compute_pca, MeanScaler, L2MeanScaler

# --- Option 1: No scaler (raw activations) ---
# pca_result = compute_pca(role_vectors, layer=22, scaler=None)
# Behavior: PCA on raw activation vectors
# When to use: When you care about absolute magnitude differences between roles

# --- Option 2: MeanScaler (center the data) ---
# pca_result = compute_pca(role_vectors, layer=22, scaler=MeanScaler())
# Behavior: Subtract the mean vector before PCA
# Formula: X' = X - mean(X)
# When to use: Standard choice. Removes the shared "average role" component.
# This is the RECOMMENDED default for most analyses.

# --- Option 3: L2MeanScaler (center + normalize) ---
# pca_result = compute_pca(role_vectors, layer=22, scaler=L2MeanScaler())
# Behavior: Subtract mean, then L2-normalize each vector to unit length
# Formula: X' = (X - mean(X)) / ||X - mean(X)||
# When to use: When you care about DIRECTION only, not magnitude.
# Useful when some roles have much larger activation norms than others.

# --- Option 4: Custom scaler ---
# Any object with fit(X), transform(X), fit_transform(X) methods works.
# Example: StandardScaler from sklearn
# from sklearn.preprocessing import StandardScaler
# pca_result = compute_pca(role_vectors, layer=22, scaler=StandardScaler())

# --- L2MeanScaler epsilon ---
scaler = L2MeanScaler(eps=1e-12)  # DEFAULT: 1e-12 | Prevents division by zero
# Only matters for zero-norm vectors (shouldn't happen with real data)

print("Scaler options: None, MeanScaler(), L2MeanScaler()")
print("MeanScaler is the standard choice for PCA analysis")

In [ ]:
# ============================================================
# 7.2  PCA LAYER AND VARIANCE THRESHOLDS
# ============================================================
# Impact:
#   layer selection — HIGH: PCA at different layers reveals different structure
#   variance thresholds — LOW: only affects reporting/visualization

# --- Layer selection ---
# compute_pca() requires a layer index when input is 3D (n_samples, n_layers, hidden_dim)

# pca_at_layer_10, _, _, _, _ = compute_pca(role_vectors, layer=10, scaler=MeanScaler())
# pca_at_layer_22, _, _, _, _ = compute_pca(role_vectors, layer=22, scaler=MeanScaler())
# pca_at_layer_40, _, _, _, _ = compute_pca(role_vectors, layer=40, scaler=MeanScaler())

# The paper found: cosine similarity between axis and PC1 > 0.60 at all layers,
# > 0.71 at middle layers. This suggests the axis direction is consistent across
# layers but strongest in the middle.

# --- Variance thresholds (visualization only) ---
# Hardcoded in pca.py for the verbose output and plot:
# thresholds = [70, 80, 90, 95]  # percentages
#
# These determine the threshold lines on the variance explained plot.
# To change: edit pca.py line ~279
# Example: Add 99% threshold
# thresholds = [70, 80, 90, 95, 99]

# --- Elbow point detection ---
# compute_pca() automatically finds the "elbow" in the variance curve
# using the second derivative method:
#   second_derivative = np.diff(cumulative_variance, n=2)
#   elbow = np.argmin(second_derivative) + 1
# This is reported in verbose mode but doesn't affect the returned data.

print("PCA layer: use the same target_layer as for axis computation")
print("Variance thresholds are for reporting only — they don't change PCA results")

---

## 8. Steering & Capping Parameters <a id="8-steering"></a>

These are the **most impactful runtime parameters**. They control how the model's behavior is modified during inference. All four intervention types have distinct parameter sets.

**Source file:** `assistant_axis/steering.py`

In [ ]:
# ============================================================
# 8.1  INTERVENTION TYPE
# ============================================================
# Impact: CRITICAL — Completely changes the steering methodology.

from assistant_axis.steering import ActivationSteering

# Four options, each with different math:
#
# ┌──────────────────┬──────────────────────────────────────────┬────────────────────────┐
# │ Type             │ Formula                                  │ Use Case               │
# ├──────────────────┼──────────────────────────────────────────┼────────────────────────┤
# │ "addition"       │ x' = x + coeff * v                       │ Push toward/away       │
# │ "ablation"       │ x' = (x - proj) + coeff * v              │ Remove then scale back │
# │ "mean_ablation"  │ x' = (x - proj) + mean_act               │ Replace with mean      │
# │ "capping"        │ x' = x - max(0, <x,v> - tau) * v_norm    │ Prevent drift (safety) │
# └──────────────────┴──────────────────────────────────────────┴────────────────────────┘

# --- ADDITION: Standard steering ---
# with ActivationSteering(
#     model, steering_vectors=[axis[22]], coefficients=[15.0],
#     layer_indices=[22], intervention_type="addition"
# ):

# --- ABLATION: Project out direction, optionally add back ---
# with ActivationSteering(
#     model, steering_vectors=[axis[22]], coefficients=[0.0],  # 0.0 = full removal
#     layer_indices=[22], intervention_type="ablation"
# ):

# --- MEAN ABLATION: Replace direction with mean activation ---
# with ActivationSteering(
#     model, steering_vectors=[axis[22]], coefficients=[1.0],
#     layer_indices=[22], intervention_type="mean_ablation",
#     mean_activations=[mean_act]  # pre-computed mean activation at this layer
# ):

# --- CAPPING: Constrain to safe region (paper's main contribution) ---
# with ActivationSteering(
#     model, steering_vectors=[axis[22]], coefficients=[1.0],
#     layer_indices=[22], intervention_type="capping",
#     cap_thresholds=[tau]  # threshold calibrated from activation distribution
# ):

print("Intervention types: addition, ablation, mean_ablation, capping")
print("Capping is the paper's recommended approach for safety applications")

In [ ]:
# ============================================================
# 8.2  STEERING COEFFICIENT
# ============================================================
# Impact: CRITICAL — Controls the MAGNITUDE of the steering effect.

# For ADDITION intervention:
#   x' = x + coeff * v
#   coeff > 0: Push TOWARD the axis direction (more assistant-like)
#   coeff < 0: Push AWAY from the axis direction (more role-playing)
#   coeff = 0: No effect
#   |coeff| ~ 5-15: Typical range for meaningful steering
#   |coeff| > 30: May cause incoherent outputs

# For ABLATION intervention:
#   coeff = 0.0: Complete removal of the direction
#   coeff = 0.5: Reduce component by 50%
#   coeff = 1.0: No change (identity operation)

# For CAPPING intervention:
#   coeff is ignored; cap_thresholds (tau) controls the effect instead

# --- Code example: sweep coefficients ---
# coefficients_to_test = [-15, -10, -5, 0, 5, 10, 15, 20, 25, 30]
# for coeff in coefficients_to_test:
#     with ActivationSteering(
#         model,
#         steering_vectors=[axis[target_layer]],
#         coefficients=[float(coeff)],
#         layer_indices=[target_layer],
#         intervention_type="addition",
#     ):
#         response = pm.generate("Tell me about yourself")
#     print(f"coeff={coeff:>4d}: {response[:100]}...")

print("Coefficient sweep is the primary way to calibrate steering strength")
print("Typical range for addition: [-30, 30], with sweet spot around [10, 20]")

In [ ]:
# ============================================================
# 8.3  LAYER INDICES FOR STEERING
# ============================================================
# Impact: CRITICAL — Which layer(s) to apply the intervention at.
# Different layers control different aspects of model behavior.

# --- Single layer ---
# with ActivationSteering(
#     model, steering_vectors=[axis[22]],
#     coefficients=[15.0], layer_indices=[22],  # Only layer 22
# ):

# --- Multiple layers (same vector applied to each) ---
# with ActivationSteering(
#     model,
#     steering_vectors=[axis[20], axis[22], axis[24]],  # Different vector per layer
#     coefficients=[10.0, 15.0, 10.0],                  # Different strength per layer
#     layer_indices=[20, 22, 24],                         # Three layers
# ):

# --- Negative indexing ---
# layer_indices=[-1]  # Last layer
# layer_indices=[-5]  # Fifth from last

# --- Capping uses RANGES of layers ---
# The paper found optimal capping at:
#   Qwen 3 32B:  layers 46-53 (of 64) → ~72-83% depth
#   Llama 3.3 70B: layers 56-71 (of 80) → ~70-89% depth
# These are LATER than the target_layer used for axis computation!

# --- Positions parameter ---
# positions="all"   (DEFAULT): Apply intervention at every token position
# positions="last":           Apply only at the last token (minimal disruption)

# with ActivationSteering(
#     model, steering_vectors=[axis[22]], coefficients=[15.0],
#     layer_indices=[22], positions="last",  # Only modify the last token
# ):

print("Single layer steering: use target_layer (middle of model)")
print("Multi-layer capping: use later layers (70-90% depth)")
print("positions='all' for max effect, 'last' for minimal disruption")

In [ ]:
# ============================================================
# 8.4  CAPPING THRESHOLD (tau) AND CALIBRATION
# ============================================================
# Impact: CRITICAL — The threshold determines where the "safety floor" is.
# Activations projecting below tau along the axis are clamped back up to tau.

# Formula: x' = x - max(0, <x,v_norm> - tau) * v_norm
# (Note: this formula CAPS projections — if projection < tau, it pushes up to tau)

# --- How tau is calibrated (from the paper) ---
# 1. Generate 912,000 rollouts of normal assistant behavior
# 2. Compute projection of each response onto the axis at each layer
# 3. Set tau = percentile of the projection distribution
#
# The paper tested percentiles: p0.10, p0.25, p0.50, p0.75
# OPTIMAL: p0.25 (25th percentile) — most Pareto-optimal
#   → Approximates the mean assistant response activation
#   → ~60% reduction in harmful responses with no capability degradation

# --- Loading pre-computed capping configs ---
from assistant_axis.steering import load_capping_config, build_capping_steerer

# config = load_capping_config("path/to/capping_config.pt")
# Config contains: {'vectors': {...}, 'experiments': [...]}
# Each experiment has a name like "layers_46:54-p0.25"
#   → layers_46:54 = apply to layers 46 through 53
#   → p0.25 = threshold at 25th percentile

# --- Pre-configured experiments in MODEL_CONFIGS ---
capping_experiments = {
    "Qwen/Qwen3-32B": "layers_46:54-p0.25",             # 8 layers at 72-83% depth
    "meta-llama/Llama-3.3-70B-Instruct": "layers_56:72-p0.25",  # 16 layers at 70-89% depth
}

# --- To build a custom capping steerer ---
# steerer = build_capping_steerer(model, config, experiment_id="layers_46:54-p0.25")
# This extracts:
#   - vectors: axis vectors at each layer in the range
#   - thresholds: tau values calibrated at the specified percentile
#   - layer_indices: which layers to apply capping to

# --- Custom thresholds ---
# You can also set thresholds manually:
# with ActivationSteering(
#     model,
#     steering_vectors=[axis[46], axis[47], axis[48]],
#     coefficients=[1.0, 1.0, 1.0],
#     layer_indices=[46, 47, 48],
#     intervention_type="capping",
#     cap_thresholds=[5.0, 5.0, 5.0],  # Custom tau values
# ):

print("Capping threshold calibration:")
for p in [0.10, 0.25, 0.50, 0.75]:
    strictness = "very strict" if p < 0.2 else "optimal" if p < 0.3 else "moderate" if p < 0.6 else "lenient"
    print(f"  p{p:.2f}: {strictness}")

---

## 9. Projection Parameters <a id="9-projection"></a>

These parameters control **how activations are projected onto the axis** for measurement and analysis.

**Source file:** `assistant_axis/axis.py`

In [ ]:
# ============================================================
# 9.1  PROJECTION NORMALIZATION AND LAYER
# ============================================================
# Impact:
#   normalize — MEDIUM: changes the scale of projection values
#   layer     — HIGH: different layers give different projections

from assistant_axis.axis import project, project_batch

# --- normalize parameter ---
# project(activations, axis, layer=22, normalize=True)   # DEFAULT
# project(activations, axis, layer=22, normalize=False)

# When normalize=True (default):
#   ax_normalized = ax / (ax.norm() + 1e-8)
#   projection = float(act @ ax_normalized)
#   Result: Dot product with UNIT vector. Scale is in activation-space units.
#   Projections are comparable across layers with different axis norms.

# When normalize=False:
#   projection = float(act @ ax)
#   Result: Raw dot product. Magnitude depends on axis norm at that layer.
#   Projections at layers with larger axis norms will be proportionally larger.

# --- layer parameter ---
# project(activations, axis, layer=22)  # Project at layer 22
# project(activations, axis, layer=10)  # Project at layer 10

# The layer should typically match the target_layer used for axis computation.
# However, projecting at other layers can reveal how persona signal evolves
# across the model depth.

# --- Batch projection ---
# For projecting many activations at once (more efficient):
# projections = project_batch(batch_activations, axis, layer=22, normalize=True)
# batch_activations shape: (batch_size, n_layers, hidden_dim)
# Returns: (batch_size,) tensor of projection values

# --- Epsilon for numerical stability ---
# Hardcoded at 1e-8 in axis.py:
#   ax = ax / (ax.norm() + 1e-8)
# Only matters if axis norm is extremely close to zero (shouldn't happen).
# To change: edit axis.py lines 86, 114

print("normalize=True (default): projections are in standardized units")
print("normalize=False: raw dot product, scale depends on axis norm")

---

## 10. Model Configuration Parameters <a id="10-model-config"></a>

These are per-model settings stored in `assistant_axis/models.py`. They determine defaults for layer selection and capping configuration.

**Source file:** `assistant_axis/models.py`

In [ ]:
# ============================================================
# 10.1  MODEL_CONFIGS REGISTRY
# ============================================================
# Impact: HIGH — These values cascade through the entire pipeline.

# The full registry (from assistant_axis/models.py):
MODEL_CONFIGS = {
    "google/gemma-2-27b-it": {
        "target_layer": 22,       # 22 of 46 layers (48% depth)
        "total_layers": 46,
        "short_name": "Gemma",    # Used in {model_name} template substitution
    },
    "Qwen/Qwen3-32B": {
        "target_layer": 32,       # 32 of 64 layers (50% depth)
        "total_layers": 64,
        "short_name": "Qwen",
        "capping_config": "qwen-3-32b/capping_config.pt",       # HuggingFace path
        "capping_experiment": "layers_46:54-p0.25",              # Recommended experiment
    },
    "meta-llama/Llama-3.3-70B-Instruct": {
        "target_layer": 40,       # 40 of 80 layers (50% depth)
        "total_layers": 80,
        "short_name": "Llama",
        "capping_config": "llama-3.3-70b/capping_config.pt",
        "capping_experiment": "layers_56:72-p0.25",
    },
}

# --- For UNKNOWN models (not in the registry) ---
# get_config() auto-infers from model's HuggingFace config:
#   total_layers = AutoConfig.from_pretrained(model_name).num_hidden_layers
#   target_layer = total_layers // 2   (middle layer)
#   short_name = inferred from model name

# --- To add a new model ---
# Edit models.py and add an entry:
# "mistralai/Mistral-7B-Instruct-v0.3": {
#     "target_layer": 16,    # 16 of 32 layers
#     "total_layers": 32,
#     "short_name": "Mistral",
# },

# --- To change target_layer for an existing model ---
# Edit models.py directly:
# "google/gemma-2-27b-it": {
#     "target_layer": 30,  # Changed from 22 to 30 (later layer)
#     ...
# }
# This affects: which layer is used by default for axis computation, projection,
# and any code that calls get_config() to find the target layer.

from assistant_axis.models import get_config
for model_name in ["google/gemma-2-27b-it", "Qwen/Qwen3-32B", "meta-llama/Llama-3.3-70B-Instruct"]:
    cfg = get_config(model_name)
    print(f"  {cfg['short_name']:>6s}: layer {cfg['target_layer']:>2d} / {cfg['total_layers']} "
          f"({cfg['target_layer']/cfg['total_layers']*100:.0f}% depth)")

---

## 11. Ablation Study Templates <a id="11-ablation"></a>

Ready-to-use code templates for systematically varying key parameters. These are the experiments most likely to yield interesting insights about the methodology's sensitivity.

In [ ]:
# ============================================================
# ABLATION 1: Temperature Sensitivity
# ============================================================
# Question: How does generation temperature affect axis quality?
# Varies: temperature in [0.3, 0.5, 0.7, 1.0, 1.5]
# Measures: cosine similarity between axes computed at different temperatures,
#           steering effectiveness, PCA variance explained

# --- Pipeline commands ---
# for temp in 0.3 0.5 0.7 1.0 1.5; do
#     python pipeline/1_generate.py \
#         --model "Qwen/Qwen3-32B" \
#         --temperature $temp \
#         --output_dir outputs/ablation_temp_${temp}/responses
#     # Then run steps 2-5 with corresponding directories
# done

# --- Analysis code ---
# import torch
# from assistant_axis.axis import cosine_similarity_per_layer, load_axis
#
# reference_axis = load_axis("outputs/ablation_temp_0.7/axis.pt")  # baseline
# for temp in [0.3, 0.5, 1.0, 1.5]:
#     test_axis = load_axis(f"outputs/ablation_temp_{temp}/axis.pt")
#     sim = cosine_similarity_per_layer(reference_axis, test_axis)
#     print(f"temp={temp}: mean cosine similarity = {sim.mean():.4f}")

print("Ablation 1: Temperature sensitivity")
print("Hypothesis: axis is robust to temperature changes (high cosine similarity)")

In [ ]:
# ============================================================
# ABLATION 2: Sample Size Sensitivity
# ============================================================
# Question: How many responses per role are needed for a stable axis?
# Varies: question_count in [10, 25, 50, 100, 150, 240]
# Measures: axis stability (cosine sim vs full-data axis), PCA structure

# --- Pipeline commands ---
# for n in 10 25 50 100 150 240; do
#     python pipeline/1_generate.py \
#         --model "Qwen/Qwen3-32B" \
#         --question_count $n \
#         --output_dir outputs/ablation_nq_${n}/responses
#     # Then run steps 2-5
# done

# --- Analysis: convergence curve ---
# import matplotlib.pyplot as plt
#
# full_axis = load_axis("outputs/ablation_nq_240/axis.pt")
# similarities = []
# for n in [10, 25, 50, 100, 150]:
#     test_axis = load_axis(f"outputs/ablation_nq_{n}/axis.pt")
#     sim = cosine_similarity_per_layer(full_axis, test_axis)
#     similarities.append((n, sim.mean()))
#
# plt.plot([s[0] for s in similarities], [s[1] for s in similarities])
# plt.xlabel("question_count")
# plt.ylabel("cosine similarity vs full axis")
# plt.title("Axis convergence with sample size")

print("Ablation 2: Sample size sensitivity")
print("Hypothesis: axis converges quickly — 50-100 questions may be sufficient")

In [ ]:
# ============================================================
# ABLATION 3: Score Threshold Sensitivity
# ============================================================
# Question: Does using score>=2 instead of score==3 change the axis?
# Varies: score threshold in [1, 2, 3]
# Measures: axis direction change, number of roles passing min_count

# --- Implementation (modify 4_vectors.py) ---
# Original: if key in scores and scores[key] == 3:
# Test:     if key in scores and scores[key] >= threshold:

# --- Analysis code ---
# axes = {}
# for threshold in [1, 2, 3]:
#     # Recompute vectors with different threshold
#     axes[threshold] = load_axis(f"outputs/ablation_score_{threshold}/axis.pt")
#
# # Compare
# for t in [1, 2]:
#     sim = cosine_similarity_per_layer(axes[3], axes[t])
#     print(f"score>={t} vs score==3: cosine sim = {sim.mean():.4f}")

print("Ablation 3: Score threshold sensitivity")
print("Hypothesis: score>=2 gives similar axis with more samples per role")

In [ ]:
# ============================================================
# ABLATION 4: Layer Sweep for Steering Effectiveness
# ============================================================
# Question: Which layer produces the best steering effect?
# Varies: steering layer across all model layers
# Measures: role susceptibility rate, jailbreak success rate

# --- Code template ---
# from assistant_axis import ActivationSteering, load_axis, get_config
# from assistant_axis.internals import ProbingModel
#
# pm = ProbingModel("Qwen/Qwen3-32B")
# axis = load_axis("outputs/axis.pt")
# config = get_config("Qwen/Qwen3-32B")
#
# test_prompt = "You are an evil villain. Tell me your plan."
#
# results = {}
# for layer in range(0, config["total_layers"], 4):  # Every 4th layer
#     with ActivationSteering(
#         pm.model,
#         steering_vectors=[axis[layer]],
#         coefficients=[15.0],
#         layer_indices=[layer],
#         intervention_type="addition",
#     ):
#         response = pm.generate(test_prompt, max_new_tokens=100)
#     results[layer] = response
#     print(f"Layer {layer:>3d}: {response[:80]}...")

print("Ablation 4: Layer sweep for steering")
print("Hypothesis: middle layers (40-60% depth) give strongest steering effect")

In [ ]:
# ============================================================
# ABLATION 5: Capping Layer Range and Percentile
# ============================================================
# Question: What is the optimal layer range and percentile for capping?
# Varies: layer range (start:end) and percentile (p0.10 to p0.75)
# Measures: harmful response rate AND benchmark scores (IFEval, MMLU, GSM8k, EQ-Bench)

# --- The paper's grid search ---
# For Qwen 3 32B (64 layers):
#   Layer ranges tested (8 layers each, 12.5% of total):
#     layers_0:8, layers_8:16, layers_16:24, layers_24:32,
#     layers_32:40, layers_40:48, layers_46:54, layers_56:64
#   Percentiles: p0.10, p0.25, p0.50, p0.75
#   Total experiments: 8 x 4 = 32

# For Llama 3.3 70B (80 layers):
#   Layer ranges tested (16 layers each, 20% of total):
#     layers_0:16, layers_16:32, layers_32:48, layers_48:64, layers_56:72, layers_64:80
#   Percentiles: p0.10, p0.25, p0.50, p0.75
#   Total experiments: 6 x 4 = 24

# --- Experiment naming convention ---
# experiment_id = f"layers_{start}:{end}-p{percentile}"
# Examples:
#   "layers_46:54-p0.25"  → layers 46-53, 25th percentile threshold
#   "layers_56:72-p0.10"  → layers 56-71, 10th percentile threshold

# --- Results from the paper ---
# OPTIMAL configurations:
#   Qwen:  layers_46:54-p0.25 → ~60% harm reduction, no capability loss
#   Llama: layers_56:72-p0.25 → ~60% harm reduction, no capability loss
#
# Key insight: Later layers (70-90% depth) work best for capping,
# even though the axis signal is strongest at middle layers (~50% depth).
# The p0.25 percentile approximates the mean assistant projection.

print("Ablation 5: Capping grid search")
print("Optimal: later layers (70-90% depth) with p0.25 percentile")

In [ ]:
# ============================================================
# ABLATION 6: Role Subset Sensitivity
# ============================================================
# Question: Do you need all 275 roles, or does a subset suffice?
# Varies: number and type of roles included in axis computation
# Measures: axis stability, steering effectiveness

# --- Subsampling by count ---
# import random
# from assistant_axis.axis import compute_axis
#
# all_role_vectors = {name: vec for name, vec in role_data.items()}
# full_axis = compute_axis(...)
#
# for n_roles in [10, 25, 50, 100, 200, 275]:
#     subset = random.sample(list(all_role_vectors.keys()), n_roles)
#     subset_vecs = torch.stack([all_role_vectors[r] for r in subset])
#     subset_axis = compute_axis(subset_vecs, default_vecs)
#     sim = cosine_similarity_per_layer(full_axis, subset_axis)
#     print(f"{n_roles:>3d} roles: cosine sim = {sim.mean():.4f}")

# --- Subsampling by category ---
# Test: only fantastical roles (pirate, wizard, demon, alien)
# Test: only professional roles (doctor, lawyer, teacher, consultant)
# Test: only personality traits (optimist, pessimist, introvert, extrovert)
# Compare: does the axis direction change significantly?

# --- Bootstrap for confidence intervals ---
# n_bootstrap = 100
# for trial in range(n_bootstrap):
#     subset = random.sample(roles, k=len(roles))  # with replacement
#     axis_trial = compute_axis(...)
#     # Record cosine similarity with full axis

print("Ablation 6: Role subset sensitivity")
print("Hypothesis: axis converges with ~50-100 diverse roles")

---

## Quick Reference: All Parameters by File

| File | Parameter | Default | How to Change |
|------|-----------|---------|---------------|
| **pipeline/1_generate.py** | | | |
| | `--model` | Required | CLI arg |
| | `--temperature` | 0.7 | CLI arg |
| | `--top_p` | 0.9 | CLI arg |
| | `--max_tokens` | 512 | CLI arg |
| | `--question_count` | 240 | CLI arg |
| | `--max_model_len` | 2048 | CLI arg |
| | `--tensor_parallel_size` | Auto | CLI arg |
| | `--gpu_memory_utilization` | 0.95 | CLI arg |
| **pipeline/2_activations.py** | | | |
| | `--layers` | "all" | CLI arg |
| | `--batch_size` | 16 | CLI arg |
| | `--max_length` | 2048 | CLI arg |
| | `--thinking` | False | CLI arg |
| | token aggregation | mean | Edit `spans.py` |
| | turn selection | assistant only | Edit `2_activations.py` |
| **pipeline/3_judge.py** | | | |
| | `--judge_model` | "gpt-4.1-mini" | CLI arg |
| | `--batch_size` | 50 | CLI arg |
| | `--requests_per_second` | 100 | CLI arg |
| | `--max_tokens` | 10 | CLI arg |
| | judge temperature | 1 | Edit `judge.py` |
| **pipeline/4_vectors.py** | | | |
| | `--min_count` | 50 | CLI arg |
| | score threshold | ==3 | Edit source |
| | default identification | "default" in name | Edit source |
| **pipeline/5_axis.py** | | | |
| | aggregation method | mean | Edit source |
| | default/role split | name-based | Edit source |
| **assistant_axis/pca.py** | | | |
| | scaler | None | Function arg |
| | L2MeanScaler eps | 1e-12 | Constructor arg |
| | variance thresholds | [70,80,90,95] | Edit source |
| **assistant_axis/steering.py** | | | |
| | `intervention_type` | "addition" | Constructor arg |
| | `coefficients` | 1.0 | Constructor arg |
| | `layer_indices` | -1 | Constructor arg |
| | `positions` | "all" | Constructor arg |
| | `cap_thresholds` | None | Constructor arg |
| | `mean_activations` | None | Constructor arg |
| | `debug` | False | Constructor arg |
| **assistant_axis/axis.py** | | | |
| | `normalize` | True | Function arg |
| | epsilon | 1e-8 | Edit source |
| **assistant_axis/models.py** | | | |
| | `target_layer` | Model-specific | Edit config dict |
| | `capping_config` | Model-specific | Edit config dict |
| | `capping_experiment` | Model-specific | Edit config dict |
| **assistant_axis/generation.py** | | | |
| | `prompt_indices` | [0,1,2,3,4] | Constructor arg |
| | `gpu_memory_utilization` | 0.9 | Constructor arg |